# Clase 16 — Decoradores, Generadores y Context Managers

En esta clase exploraremos tres técnicas avanzadas de Python: **decoradores** (modificar funciones), **generadores** (evaluación perezosa) y **context managers** (gestión de recursos).

**Contenido de esta clase:**
1. Funciones como objetos
2. Decoradores básicos
3. Decoradores con argumentos
4. `functools.wraps`
5. Decoradores de clase (`@staticmethod`, `@classmethod`, `@property`)
6. Generadores con `yield`
7. Protocolo de iteración
8. Context managers
9. `contextlib`
10. Ejercicio práctico


## 1. Funciones como objetos

En Python, las funciones son **objetos de primera clase**: se pueden asignar a variables, pasar como argumentos y retornar desde otras funciones.


In [ ]:
# Asignar función a variable
def saludar(nombre):
    return f"Hola, {nombre}!"

mi_funcion = saludar
print(mi_funcion("Python"))

# Pasar función como argumento
def aplicar(func, valor):
    return func(valor)

print(aplicar(str.upper, "python"))


In [ ]:
# Función que retorna otra función
def crear_saludo(formal):
    if formal:
        return lambda nombre: f"Buenos días, Sr./Sra. {nombre}"
    else:
        return lambda nombre: f"¡Hola, {nombre}!"

saludo_formal = crear_saludo(True)
saludo_informal = crear_saludo(False)

print(saludo_formal("García"))
print(saludo_informal("García"))


## 2. Decoradores básicos

Un **decorador** es una función que toma otra función, la envuelve, y retorna una versión modificada.


In [ ]:
# Decorador simple: medir tiempo de ejecución
import time

def medir_tiempo(func):
    """Decorador que mide el tiempo de ejecución."""
    def wrapper(*args, **kwargs):
        inicio = time.time()
        resultado = func(*args, **kwargs)
        fin = time.time()
        print(f"{func.__name__} tardó {fin - inicio:.4f} segundos")
        return resultado
    return wrapper

# Aplicar el decorador
@medir_tiempo
def operacion_lenta():
    """Simula una operación que toma tiempo."""
    time.sleep(0.5)
    return "Listo"

resultado = operacion_lenta()


In [ ]:
# Decorador con argumentos
@medir_tiempo
def saludar(nombre, saludo="Hola"):
    return f"{saludo}, {nombre}!"

print(saludar("Ana", saludo="Buenos días"))


## 3. Decoradores con argumentos

A veces queremos que el decorador mismo reciba parámetros.


In [ ]:
# Decorador que se repite N veces
def repetir(veces):
    """Decorador que ejecuta la función N veces."""
    def decorador(func):
        def wrapper(*args, **kwargs):
            resultados = []
            for i in range(veces):
                resultados.append(func(*args, **kwargs))
            return resultados
        return wrapper
    return decorador

# Aplicar
@repetir(3)
def saludar(nombre):
    return f"Hola, {nombre}!"

resultados = saludar("Python")
print(f"Resultados: {resultados}")


## 4. `functools.wraps`

Por defecto, el decorador reemplaza el nombre y docstring de la función. `functools.wraps` preserva la metadata original.


In [ ]:
from functools import wraps

# Sin wraps
def decorador_sin_wraps(func):
    def wrapper(*args, **kwargs):
        """Wrapper docstring"""
        return func(*args, **kwargs)
    return wrapper

@decorador_sin_wraps
def mi_funcion():
    """Documentación original"""
    pass

print(f"Sin wraps - nombre: {mi_funcion.__name__}")
print(f"Sin wraps - doc: {mi_funcion.__doc__}")


In [ ]:
# Con wraps
def decorador_con_wraps(func):
    @wraps(func)  # Preserva la metadata
    def wrapper(*args, **kwargs):
        """Wrapper docstring"""
        return func(*args, **kwargs)
    return wrapper

@decorador_con_wraps
def mi_funcion():
    """Documentación original"""
    pass

print(f"\nCon wraps - nombre: {mi_funcion.__name__}")
print(f"Con wraps - doc: {mi_funcion.__doc__}")


## 5. Decoradores de clase

Ya vimos `@staticmethod`, `@classmethod` y `@property`. Aquí un ejemplo más completo:


In [ ]:
class Producto:
    """Producto con decoradores de clase."""
    
    IVA = 0.16  # Atributo de clase
    
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self._precio = precio  # "privado"
    
    @property
    def precio(self):
        """Getter del precio."""
        return self._precio
    
    @precio.setter
    def precio(self, valor):
        if valor < 0:
            raise ValueError("Precio no puede ser negativo")
        self._precio = valor
    
    @property
    def precio_con_iva(self):
        """Precio con IVA incluido."""
        return self._precio * (1 + self.IVA)
    
    @staticmethod
    def validar_precio(precio):
        """Valida que un precio sea válido."""
        return precio >= 0
    
    @classmethod
    def desde_string(cls, texto):
        """Crea un producto desde un string 'nombre:precio'."""
        nombre, precio = texto.split(':')
        return cls(nombre, float(precio))

# Uso
p = Producto("Laptop", 1000)
print(f"Precio: ${p.precio}")
print(f"Precio con IVA: ${p.precio_con_iva:.2f}")
print(f"¿500 es válido? {Producto.validar_precio(500)}")

# Crear desde string (classmethod)
p2 = Producto.desde_string("Mouse:25.5")
print(f"\nDesde string: {p2.nombre} - ${p2.precio}")


## 6. Generadores con `yield`

Los **generadores** son funciones que producen valores **uno a la vez** usando `yield`.


In [ ]:
# Generador básico
def contar_hasta(n):
    """Genera números del 1 al n."""
    i = 1
    while i <= n:
        yield i
        i += 1

# Usar el generador
for numero in contar_hasta(5):
    print(numero)


In [ ]:
# Generador infinito (con límite externo)
def fibonacci_infinito():
    """Genera la secuencia de Fibonacci infinitamente."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Usar con takewhile
from itertools import islice
primeros_10 = list(islice(fibonacci_infinito(), 10))
print(f"Primeros 10 Fibonacci: {primeros_10}")


In [ ]:
# yield con envío de datos: send()
def eco():
    """Generador que eco: recibe y emite el mismo valor."""
    while True:
        valor = yield
        yield valor * 2

gen = eco()
next(gen)  # Iniciar el generador
print(gen.send(5))  # Envía 5, recibe 10
print(gen.send(10))  # Envía 10, recibe 20


## 7. Protocolo de iteración

Para que un objeto sea iterable, debe implementar `__iter__` (que retorna un iterador) o `__getitem__`. El iterador debe implementar `__iter__` y `__next__`.


In [ ]:
class Contador:
    """Clase iterable que cuenta hasta un máximo."""
    
    def __init__(self, maximo):
        self.maximo = maximo
    
    def __iter__(self):
        self.actual = 0
        return self
    
    def __next__(self):
        if self.actual >= self.maximo:
            raise StopIteration
        self.actual += 1
        return self.actual

# Usar
for numero in Contador(5):
    print(numero)


## 8. Context managers

Los **context managers** garantizan que los recursos se liberen correctamente. El ejemplo más común es `with open(...)`.


In [ ]:
# Context manager con clase: __enter__ y __exit__
class GestorArchivo:
    """Context manager simple para archivos."""
    
    def __init__(self, ruta, modo):
        self.ruta = ruta
        self.modo = modo
    
    def __enter__(self):
        """Se ejecuta al entrar al bloque with."""
        print(f"Abriendo {self.ruta}")
        self.archivo = open(self.ruta, self.modo)
        return self.archivo
    
    def __exit__(self, tipo_error, valor, traceback):
        """Se ejecuta al salir del bloque with."""
        print(f"Cerrando {self.ruta}")
        self.archivo.close()
        # Si retorna True, suprime la excepción
        return False

# Usar
import tempfile
import os
ruta = os.path.join(tempfile.gettempdir(), "test.txt")

with GestorArchivo(ruta, 'w') as f:
    f.write("Hola mundo")


## 9. `contextlib`

`contextlib.contextmanager` permite crear context managers con generadores (más simple que una clase).


In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def medir_tiempo_bloque(nombre):
    """Context manager que mide el tiempo de un bloque."""
    inicio = time.time()
    print(f"Iniciando {nombre}...")
    yield
    fin = time.time()
    print(f"{nombre} tardó {fin - inicio:.4f} segundos")

# Usar
with medir_tiempo_bloque("Operación"):
    time.sleep(0.3)
    # Hacer algo aquí


In [ ]:
# Context manager con manejo de excepciones
@contextmanager
def archivo_temporal():
    """Crea un archivo temporal y lo elimina al final."""
    import tempfile
    fd, ruta = tempfile.mkstemp()
    print(f"Creado: {ruta}")
    try:
        yield ruta
    finally:
        os.close(fd)
        os.unlink(ruta)
        print(f"Eliminado: {ruta}")

# Usar
with archivo_temporal() as ruta:
    with open(ruta, 'w') as f:
        f.write("Datos temporales")
    # El archivo se elimina automáticamente al salir


## 10. Ejercicio Práctico: Decoradores y Context Managers

**Contexto:** Vas a crear una pequeña librería de decoradores y context managers útiles.

**Instrucciones:**

Crea un archivo `decoradores.py` con:

1. `@reintentar(max_intentos=3, delay=1)` — Reintenta una función si falla, con delay entre intentos.
2. `@log_llamada` — Imprime el nombre de la función y sus argumentos cada vez que se llama.
3. `@medir_tiempo` — Mide y muestra el tiempo de ejecución de una función.
4. `@contexto_prueba` (context manager) — Cambia temporalmente el directorio de trabajo.
5. `@conexion_db` (context manager) — Simula una conexión a base de datos (solo abre y cierra).

**Pistas:**
- Usa `time.sleep()` para los delays.
- Usa `try/except` en el decorador de reintentos.
- Para el log, usa `functools.wraps`.

**Restricción del ejercicio:**
- No uses librerías externas.
- Solo el módulo estándar.


In [ ]:
"""
Solución al Ejercicio Práctico
Este código debe ir en un archivo decoradores.py
"""

import time
import functools
from contextlib import contextmanager

def reintentar(max_intentos=3, delay=1):
    """Decorador que reintenta una función si falla."""
    def decorador(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for intento in range(1, max_intentos + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if intento == max_intentos:
                        print(f"Falló después de {max_intentos} intentos: {e}")
                        raise
                    print(f"Intento {intento} falló: {e}. Reintentando en {delay}s...")
                    time.sleep(delay)
        return wrapper
    return decorador

def log_llamada(func):
    """Decorador que registra cada llamada."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        args_str = ', '.join(repr(a) for a in args)
        kwargs_str = ', '.join(f"{k}={v!r}" for k, v in kwargs.items())
        todos = ', '.join(filter(None, [args_str, kwargs_str]))
        print(f"Llamando: {func.__name__}({todos})")
        resultado = func(*args, **kwargs)
        print(f"  Resultado: {resultado}")
        return resultado
    return wrapper

def medir_tiempo(func):
    """Decorador que mide el tiempo de ejecución."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        inicio = time.time()
        resultado = func(*args, **kwargs)
        fin = time.time()
        print(f"{func.__name__} tardó {fin - inicio:.4f} segundos")
        return resultado
    return wrapper

@contextmanager
def directorio_temporal(ruta):
    """Context manager que cambia temporalmente el directorio."""
    directorio_original = os.getcwd()
    os.chdir(ruta)
    try:
        yield
    finally:
        os.chdir(directorio_original)

@contextmanager
def conexion_db(nombre_db):
    """Context manager que simula una conexión a BD."""
    print(f"Conectando a {nombre_db}...")
    conexion = f"<Conexion a {nombre_db}>"
    try:
        yield conexion
    finally:
        print(f"Cerrando conexión a {nombre_db}")


In [ ]:
# Probar los decoradores y context managers

@log_llamada
@medir_tiempo
def sumar(a, b):
    return a + b

@reintentar(max_intentos=3, delay=0.1)
def operacion_inestable(exito_en_intento=2):
    """Falla los primeros N-1 intentos, luego tiene éxito."""
    operacion_inestable.intento = getattr(operacion_inestable, 'intento', 0) + 1
    if operacion_inestable.intento < exito_en_intento:
        raise ValueError(f"Fallo en intento {operacion_inestable.intento}")
    return "Éxito"

# Probar decorador compuesto
print("--- sumar ---")
resultado = sumar(3, 5)

# Probar reintentos
print("\n--- operacion_inestable ---")
resultado = operacion_inestable(exito_en_intento=2)
print(f"Resultado: {resultado}")

# Probar context managers
print("\n--- directorio_temporal ---")
with directorio_temporal(tempfile.gettempdir()):
    print(f"Directorio actual: {os.getcwd()}")
print(f"Volvió a: {os.getcwd()}")

print("\n--- conexion_db ---")
with conexion_db("mi_base_datos") as conn:
    print(f"Usando: {conn}")


## Resumen de la clase

En esta clase aprendimos:

1. **Funciones como objetos:** asignar, pasar y retornar funciones.
2. **Decoradores:** funciones que envuelven otras funciones.
3. **Decoradores con argumentos:** decorador de tres niveles.
4. **`functools.wraps`:** preservar metadata de la función original.
5. **Decoradores de clase:** `@staticmethod`, `@classmethod`, `@property`.
6. **Generadores:** funciones con `yield` para evaluación perezosa.
7. **Protocolo de iteración:** `__iter__` y `__next__`.
8. **Context managers:** protocolo `__enter__`/`__exit__` con clases.
9. **`contextlib.contextmanager`:** context managers con generadores.
10. **Casos de uso:** logging, retry, timing, gestión de recursos.

**En la siguiente clase** (Clase 17) veremos **Patrones de Diseño**: Singleton, Factory, Observer, Strategy y Decorator (como patrón).
